## CS310 Natural Language Processing
## Assignment 1. Neural Network-based Text Classification

**Total points**: 30

You should roughtly follow the structure of the notebook. Add additional cells if you feel needed. 

You can (and you should) re-use the code from Lab 2. 

Make sure your code is readable and well-structured.

### 0. Import Necessary Libraries

In [ ]:
import json
import random
import re
from collections import Counter
from copy import deepcopy
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

SEED = 310
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def resolve_data_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "coding assignment" / "A1",
        Path.cwd() / "A1",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "train.jsonl").exists() and (candidate / "test.jsonl").exists():
            return candidate
    raise FileNotFoundError("Cannot find train.jsonl and test.jsonl in the current environment.")

DATA_DIR = resolve_data_dir()
print(f"Using device: {device}")
print(f"Data directory: {DATA_DIR}")
print("Advanced tokenizer: regex patterns + phrase lexicon fallback")

### 1. Data Processing

In [ ]:
CHINESE_CHAR_RE = re.compile(r"[\u4e00-\u9fff]")
CHINESE_SPAN_RE = re.compile(r"^[\u4e00-\u9fff]+$")
ADVANCED_PATTERN = re.compile(r"[A-Za-z]+|\d+|[\u4e00-\u9fff]+|[^\u4e00-\u9fffA-Za-z0-9\s]+")

def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            item = json.loads(line)
            records.append({
                "sentence": item["sentence"],
                "label": int(item["label"][0]),
                "id": item["id"],
            })
    return records

class HumorDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        item = self.records[index]
        return item["sentence"], item["label"]

class Vocabulary:
    def __init__(self, stoi):
        self.stoi = stoi
        self.itos = {index: token for token, index in stoi.items()}
        self.unk_index = stoi["<unk>"]

    def __len__(self):
        return len(self.stoi)

    def __call__(self, tokens):
        return [self.stoi.get(token, self.unk_index) for token in tokens]

def basic_char_tokenizer(text):
    return CHINESE_CHAR_RE.findall(text)

def build_phrase_lexicon(records, min_freq=8, max_word_len=4):
    counts = Counter()
    for record in records:
        chinese_text = "".join(CHINESE_CHAR_RE.findall(record["sentence"]))
        for start in range(len(chinese_text)):
            upper = min(max_word_len, len(chinese_text) - start)
            for length in range(2, upper + 1):
                counts[chinese_text[start:start + length]] += 1
    return {token for token, freq in counts.items() if freq >= min_freq}

def greedy_segment_chinese(text, lexicon, max_word_len=4):
    tokens = []
    index = 0
    while index < len(text):
        match = None
        upper = min(max_word_len, len(text) - index)
        for length in range(upper, 1, -1):
            candidate = text[index:index + length]
            if candidate in lexicon:
                match = candidate
                break
        if match is None:
            match = text[index]
        tokens.append(match)
        index += len(match)
    return tokens

train_records = read_jsonl(DATA_DIR / "train.jsonl")
test_records = read_jsonl(DATA_DIR / "test.jsonl")
phrase_lexicon = build_phrase_lexicon(train_records)

def advanced_tokenizer(text):
    tokens = []
    for piece in ADVANCED_PATTERN.findall(text):
        if CHINESE_SPAN_RE.fullmatch(piece):
            tokens.extend(greedy_segment_chinese(piece, phrase_lexicon))
        else:
            tokens.append(piece)
    return tokens

def build_vocab(records, tokenizer, min_freq=1):
    counter = Counter()
    for record in records:
        counter.update(tokenizer(record["sentence"]))
    stoi = {"<unk>": 0}
    for token, freq in counter.most_common():
        if freq >= min_freq and token not in stoi:
            stoi[token] = len(stoi)
    return Vocabulary(stoi)

full_train_dataset = HumorDataset(train_records)
test_dataset = HumorDataset(test_records)
valid_size = max(1, int(len(full_train_dataset) * 0.1))
train_size = len(full_train_dataset) - valid_size
split_generator = torch.Generator().manual_seed(SEED)
train_subset, valid_subset = random_split(full_train_dataset, [train_size, valid_size], generator=split_generator)

def make_collate_fn(tokenizer, vocab):
    def collate_batch(batch):
        labels = []
        token_ids_list = []
        offsets = [0]

        for text, label in batch:
            token_ids = vocab(tokenizer(text))
            if not token_ids:
                token_ids = [vocab.unk_index]
            token_tensor = torch.tensor(token_ids, dtype=torch.long)
            labels.append(label)
            token_ids_list.append(token_tensor)
            offsets.append(token_tensor.size(0))

        labels = torch.tensor(labels, dtype=torch.long)
        offsets = torch.tensor(offsets[:-1], dtype=torch.long).cumsum(dim=0)
        token_ids = torch.cat(token_ids_list)
        return labels.to(device), token_ids.to(device), offsets.to(device)

    return collate_batch

def build_experiment(tokenizer_name, tokenizer_fn, description, batch_size=64):
    vocab = build_vocab(train_records, tokenizer_fn)
    collate_fn = make_collate_fn(tokenizer_fn, vocab)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(valid_subset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    return {
        "name": tokenizer_name,
        "description": description,
        "tokenizer": tokenizer_fn,
        "vocab": vocab,
        "train_loader": train_loader,
        "valid_loader": valid_loader,
        "test_loader": test_loader,
    }

experiments = {
    "basic_char": build_experiment(
        "basic_char",
        basic_char_tokenizer,
        "single Chinese characters only; drop English, digits, and punctuations",
    ),
    "advanced": build_experiment(
        "advanced",
        advanced_tokenizer,
        "regex patterns plus phrase lexicon segmentation for Chinese spans",
    ),
}

sample_text = train_records[0]["sentence"]
print(f"Train size: {len(train_records)}, test size: {len(test_records)}")
print("Sample sentence:", sample_text)
print("Basic tokens:", basic_char_tokenizer(sample_text))
print("Advanced tokens:", advanced_tokenizer(sample_text))
for name, config in experiments.items():
    print(f"{name} vocabulary size: {len(config['vocab'])}")

### 2. Build the Model

In [ ]:
NUM_CLASSES = 2
EMBED_DIM = 128
HIDDEN_DIMS = (128, 64)
DROPOUT = 0.25
LEARNING_RATE = 1e-3
EPOCHS = 10

class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dims, num_classes, dropout=0.25):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean", sparse=False)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], num_classes),
        )
        self.init_weights()

    def init_weights(self):
        init_range = 0.5
        self.embedding.weight.data.uniform_(-init_range, init_range)
        for module in self.classifier:
            if isinstance(module, nn.Linear):
                module.weight.data.uniform_(-init_range, init_range)
                module.bias.data.zero_()

    def forward(self, token_ids, offsets):
        embedded = self.embedding(token_ids, offsets)
        return self.classifier(embedded)

def create_model(vocab_size):
    return TextClassificationModel(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        hidden_dims=HIDDEN_DIMS,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
    ).to(device)

demo_model = create_model(len(experiments["advanced"]["vocab"]))
print(demo_model)
print("Number of trainable parameters:", sum(parameter.numel() for parameter in demo_model.parameters() if parameter.requires_grad))

### 3. Train and Evaluate

In [ ]:
def compute_binary_metrics(y_true, y_pred):
    tp = sum(1 for gold, pred in zip(y_true, y_pred) if gold == 1 and pred == 1)
    tn = sum(1 for gold, pred in zip(y_true, y_pred) if gold == 0 and pred == 0)
    fp = sum(1 for gold, pred in zip(y_true, y_pred) if gold == 0 and pred == 1)
    fn = sum(1 for gold, pred in zip(y_true, y_pred) if gold == 1 and pred == 0)

    total = len(y_true)
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for labels, token_ids, offsets in dataloader:
        optimizer.zero_grad()
        logits = model(token_ids, offsets)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += labels.size(0)

    return {
        "loss": total_loss / total_count,
        "accuracy": total_correct / total_count,
    }

@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    y_true = []
    y_pred = []

    for labels, token_ids, offsets in dataloader:
        logits = model(token_ids, offsets)
        loss = criterion(logits, labels)
        predictions = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())

    metrics = compute_binary_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / len(y_true)
    return metrics

def run_experiment(config):
    model = create_model(len(config["vocab"]))
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1
    )

    best_state = None
    best_valid_f1 = -1.0
    history = []

    for epoch in range(1, EPOCHS + 1):
        train_metrics = train_one_epoch(model, config["train_loader"], optimizer, criterion)
        valid_metrics = evaluate(model, config["valid_loader"], criterion)
        scheduler.step(valid_metrics["f1"])

        history.append({
            "epoch": epoch,
            "train": train_metrics,
            "valid": valid_metrics,
        })

        if valid_metrics["f1"] > best_valid_f1:
            best_valid_f1 = valid_metrics["f1"]
            best_state = deepcopy(model.state_dict())

        print(
            f"[{config['name']}] epoch {epoch:02d} | "
            f"train loss {train_metrics['loss']:.4f} | "
            f"train acc {train_metrics['accuracy']:.4f} | "
            f"valid acc {valid_metrics['accuracy']:.4f} | "
            f"valid precision {valid_metrics['precision']:.4f} | "
            f"valid recall {valid_metrics['recall']:.4f} | "
            f"valid f1 {valid_metrics['f1']:.4f}"
        )

    model.load_state_dict(best_state)
    test_metrics = evaluate(model, config["test_loader"], criterion)
    return {
        "model": model,
        "history": history,
        "test_metrics": test_metrics,
        "vocab_size": len(config["vocab"]),
        "description": config["description"],
    }

results = {}
for experiment_name, experiment_config in experiments.items():
    print("=" * 90)
    print(f"Running experiment: {experiment_name} ({experiment_config['description']})")
    results[experiment_name] = run_experiment(experiment_config)

print("\nDetailed test metrics (list each experiment)")
print("-" * 90)
for name in ["basic_char", "advanced"]:
    if name not in results:
        print(f"{name}: missing")
        continue

    m = results[name]["test_metrics"]
    print(f"{name:12s} | vocab {results[name]['vocab_size']:5d}")
    print(
        f"  accuracy={m['accuracy']:.4f}, precision={m['precision']:.4f}, "
        f"recall={m['recall']:.4f}, f1={m['f1']:.4f}, loss={m['loss']:.4f}"
    )
    print(f"  tp={m['tp']}, tn={m['tn']}, fp={m['fp']}, fn={m['fn']}")

if "basic_char" in results and "advanced" in results:
    basic = results["basic_char"]["test_metrics"]
    adv = results["advanced"]["test_metrics"]

    basic_avg = (basic["accuracy"] + basic["precision"] + basic["recall"] + basic["f1"]) / 4
    adv_avg = (adv["accuracy"] + adv["precision"] + adv["recall"] + adv["f1"]) / 4

    print("\nAverage comparison (mean of accuracy, precision, recall, f1)")
    print("-" * 90)
    print(f"basic_char average = {basic_avg:.4f}")
    print(f"advanced   average = {adv_avg:.4f}")
    print(f"difference (advanced - basic_char) = {adv_avg - basic_avg:+.4f}")

    print("\nStep-by-step metric comparison")
    print("-" * 90)
    compare_items = [
        ("accuracy", "Accuracy"),
        ("precision", "Precision"),
        ("recall", "Recall"),
        ("f1", "F1"),
        ("loss", "Loss (lower is better)"),
    ]

    for key, label in compare_items:
        b = basic[key]
        a = adv[key]
        diff = a - b
        if key == "loss":
            winner = "advanced" if a < b else ("basic_char" if b < a else "tie")
            better_diff = b - a
            print(f"{label:22s} | basic_char={b:.4f} | advanced={a:.4f} | winner={winner} | gap={better_diff:+.4f}")
        else:
            winner = "advanced" if a > b else ("basic_char" if b > a else "tie")
            print(f"{label:22s} | basic_char={b:.4f} | advanced={a:.4f} | winner={winner} | gap={diff:+.4f}")

    better_avg = "advanced" if adv_avg >= basic_avg else "basic_char"
    better_f1 = "advanced" if adv["f1"] >= basic["f1"] else "basic_char"

    print("\nWhy choose this model")
    print("-" * 90)
    print(
        "1) Core criterion is F1, because humor classification needs a balance between precision and recall."
    )
    print(
        f"2) On test set, F1 winner is {better_f1}, so it gives better balanced performance."
    )
    print(
        f"3) The average of four metrics (acc/precision/recall/f1) also favors {better_avg}, "
        "which means overall quality is better, not just one single metric."
    )

best_experiment_name = max(results, key=lambda name: results[name]["test_metrics"]["f1"])
best_result = results[best_experiment_name]
save_path = DATA_DIR / f"best_{best_experiment_name}_model.pth"
torch.save(best_result["model"].state_dict(), save_path)

print("\nBest experiment (by F1):", best_experiment_name)
print("Model checkpoint saved to:", save_path)